# 원본 실험 기록
원본 파일: 배포ver1.1.ipynb

실행 출력과 메타데이터를 제거하고 Drive 경로를 /content/project로 치환했습니다. 셀 순서와 원본 코드를 보존하므로 위에서 아래로 실행이 보장되지 않습니다. 정리된 실행 흐름은 상위 폴더의 01–05 노트북을 참고하세요.


DCGAN 모델 및 학습 루프는 TensorFlow Authors의 DCGAN 튜토리얼을 바탕으로
음악 이미지에 맞게 수정한 프로젝트 코드입니다. Copyright 2019 The TensorFlow Authors.
해당 기반 코드에는 Apache License 2.0이 적용됩니다. 저장소의 THIRD_PARTY_NOTICES.md와
LICENSES/Apache-2.0.txt를 참고하세요. 공개용 정리 과정에서 경로·셀 순서·설명을 수정했습니다.

In [ ]:
#@title
!pip install midiutil
!apt install fluidsynth
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
!pip install midi2audio

import tensorflow as tf
import glob
import imageio
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
from tensorflow.keras import layers
import time
import midiutil


from IPython import display

from midi2audio import FluidSynth

from google.colab import drive, files
drive.mount('/content/drive/')

In [ ]:
#@title
def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))
    assert model.output_shape == (None, 8, 8, 256) # 주목: 배치사이즈로 None이 주어집니다.

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    assert model.output_shape == (None, 8, 8, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    # assert model.output_shape == (None, 16, 16, 64)
    # model.add(layers.BatchNormalization())
    # model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 16, 16, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 32)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())


    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 64, 64, 1)

    return model


def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[64, 64, 1]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

generator = make_generator_model()
discriminator = make_discriminator_model()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
ckpt_path = '/content/project/ckpt_sil_jeon/ckpt-80' #모델경로
ckpt = 80 #모델고유번호

In [ ]:
checkpoint.restore(ckpt_path)
n=0 #(시행횟수)

In [ ]:
n+=1 #시행횟수(만들어지는 파일의 이름이 중복됨을 방지)
noise = tf.random.normal([1, 100]) #정규분포 노이즈
# noise = tf.random.uniform([1,100], minval=-1, maxval=1, dtype=tf.int32)
# noise = tf.cast(noise, tf.float32)
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0], cmap='gray', origin='lower')

In [ ]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
target = generated_image.numpy()
target = target.reshape(64,64)
target = (target+1)/2
target = np.around(target)
plt.imshow(target[:,:], cmap='gray', origin='lower')

미디로 쓰기

In [ ]:
#@title
reshaped_data2 = target

for i in range(36):
  reshaped_data2 = np.insert(reshaped_data2,0,0, axis=0)
for i in range(12):
  reshaped_data2 = np.insert(reshaped_data2,60,0, axis=0)
for i in range(16):
  reshaped_data2 = np.insert(reshaped_data2,112,0, axis=0)


sequence = []
stack = 0
for pit, note in enumerate(reshaped_data2):
  for i,j in enumerate(note):
    if j != 0 and i != 63:
      if note[i+1] != 0:
        stack = stack + 1
        if i+1 == 63: #다음 노트가 맨 뒤 노트일때
          sequence.append({
          'pitch' : pit,
          'start_time' : (i+1-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
          stack=0
      else:
        sequence.append({
          'pitch' : pit,
          'start_time' : (i-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
        stack=0
    elif j !=0 and i == 63 and note[i-1] == 0:
      sequence.append({
          'pitch' : pit,
          'start_time' : i/4,
          'length' : 0.25,
          'velocity' : 80
        })
      stack=0


def write_midi(seq, bpm, path):
    mf = midiutil.MIDIFile(1, file_format=1)
    track = 0
    channel = 0
    mf.addTempo(track, 0, bpm)
    for i, n in enumerate(seq):
        mf.addNote(
            track,
            channel,
            int(n['pitch']),
            n['start_time'],
            n['length'],
            int(n['velocity'])
        )
    with open(path, 'wb') as outf:
        mf.writeFile(outf)

print(f'{reshaped_data2.shape}로 바꿨음!')
print('변신준비 완료!')

write_midi(sequence, 100, f'/content/ckpt-{ckpt}_{n}.mid')

print('미디로 씀!')

들어보기

In [ ]:
fs = FluidSynth(sound_font='./font.sf2')
fs.midi_to_audio(f'/content/ckpt-{ckpt}_{n}.mid', f'/content/ckpt-{ckpt}_{n}.wav') 
display.Audio(f'/content/ckpt-{ckpt}_{n}.wav')

##다운로드 하기

In [ ]:
files.download(f'/content/ckpt-{ckpt}_{n}.mid')